# Calcusol · R1 · Comparación de modelos (y búsqueda de hiperparámetros)

Este cuaderno **ejecuta lo mismo que `src/calcusol/ml/comparar_modelos.py` y `buscar_hiperparametros.py`**, pero pensado para correrlo en una máquina con más RAM y, si la hay, GPU (por ejemplo Google Colab). No duplica código: importa los módulos del paquete `calcusol`, así que las variables, la partición y la validación son exactamente las del entrenamiento.

**Qué compara:** NASA sin corregir (B-0), una recta global (B-1), Ridge, Lasso, Random Forest, XGBoost y LightGBM, con las 17 variables de `ml/features.py`, el objetivo `residuo = Valor − ALLSKY` y validación cruzada de 5 pliegues agrupada por Sitio (solo años ≤ 2020; el test final no se toca). Reporta MAE horario, MAE mensual, RMSE, sesgo y la desviación entre pliegues.

**Qué necesitas subir a Google Drive** (en una carpeta, por ejemplo `MyDrive/calcusol/`):
1. La carpeta `src/` del repositorio (con `src/calcusol/...`).
2. `dataset_modelo.parquet` (el que está en `data/gold/` del repositorio).

**GPU:** en Colab, *Entorno de ejecución → Cambiar tipo → GPU*. Solo XGBoost usa la GPU (CUDA); LightGBM y Random Forest corren en CPU, donde ayudan la RAM y los núcleos.


In [ ]:
# Paso 0 · Instalar dependencias (en Colab; en tu PC ya están si usas el entorno del proyecto)
!pip install -q pvlib pyarrow scikit-learn lightgbm xgboost optuna pyyaml openpyxl

In [ ]:
# Paso 1 · Parámetros (cámbialos aquí)
MUESTRA   = 1.0      # fracción de las filas de ENTRENAMIENTO de cada pliegue (1.0 = todas; 0.25 = más rápido)
MODELOS   = ['Ridge', 'Lasso', 'Random Forest', 'XGBoost', 'LightGBM']
USAR_GPU  = True     # XGBoost en GPU (si no hay GPU, pon False)
TRIALS    = 25       # pruebas de Optuna (Paso 4)
MUESTRA_BUSQUEDA = 0.25

# Carpeta en Drive con src/ y dataset_modelo.parquet (solo se usa en Colab)
CARPETA_DRIVE = '/content/drive/MyDrive/calcusol'

In [ ]:
# Paso 2 · Preparar rutas (Colab: monta Drive; local: busca la raíz del proyecto)
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path(CARPETA_DRIVE)
    RUTA_PARQUET = str(RAIZ / 'dataset_modelo.parquet')
else:
    RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'calcusol').is_dir())
    RUTA_PARQUET = str(RAIZ / 'data' / 'gold' / 'dataset_modelo.parquet')
sys.path.insert(0, str(RAIZ / 'src'))

assert Path(RUTA_PARQUET).exists(), f'No encuentro el parquet: {RUTA_PARQUET}'
from calcusol.ml import comparar_modelos as cm, buscar_hiperparametros as bh
print('Raíz:', RAIZ)
print('Datos:', RUTA_PARQUET)
print('Resultados en:', cm.SALIDA)

import subprocess
print(subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout or 'Sin GPU detectada: pon USAR_GPU = False')

## Paso 3 · Comparar las familias de modelos
Para cada pliegue se entrena el modelo con los sitios de entrenamiento y se mide en los sitios que no vio. Si dos modelos difieren menos que la desviación entre pliegues, es un **empate técnico** (así se justifica elegir LightGBM por velocidad y facilidad de despliegue). Random Forest es el más lento.

In [ ]:
res = cm.ejecutar(muestra=MUESTRA, modelos=MODELOS, ruta=RUTA_PARQUET, gpu=USAR_GPU)
res[['MAE_hora', 'MAE_hora_desv', 'MAE_mes', 'MAE_mes_desv', 'RMSE_hora', 'MBE_hora', 'mejora_mes_pct', 'segundos']]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.bar(res.index, res['MAE_mes'], yerr=res['MAE_mes_desv'], capsize=4, color='#2563C9', edgecolor='white')
ax.set_ylabel('MAE mensual [Wh/m²]'); ax.set_title('Error mensual por modelo (validación cruzada por Sitio, media ± desv. entre pliegues)')
plt.xticks(rotation=20, ha='right'); plt.tight_layout()
plt.savefig(cm.SALIDA / 'comparacion_modelos.png', dpi=200); plt.show()

## Paso 4 · (Opcional) Búsqueda de hiperparámetros con Optuna
Busca los ajustes de LightGBM que minimizan el MAE mensual con la misma validación cruzada. **No cambia el modelo**: deja los mejores valores en `hiperparametros_optuna.json` para compararlos con los de `ml/train.py`.

In [ ]:
resultado = bh.ejecutar(trials=TRIALS, muestra=MUESTRA_BUSQUEDA, ruta=RUTA_PARQUET)
resultado

## Paso 5 · Llevar los resultados al repositorio
Descarga (o copia desde Drive) la carpeta `reports/ml_comparacion/` y pégala en `reports/ml_comparacion/` del repositorio:
`comparacion_modelos.csv`, `comparacion_por_pliegue.csv`, `comparacion_modelos.png`, `hiperparametros_optuna.json` y `hiperparametros_optuna_pruebas.csv`. Esas cifras son las que respaldan la sección de selección de modelos del informe.